In [3]:
#!uv add plotly

In [4]:
from aml_working_paper.routing.routing_investigate import build_candidate_windows, select_investigations
from aml_working_paper.routing.routing import extract_posterior_draws, filter_posterior_draws, sequence_results_to_dataframe, build_routing_dataframe
from aml_working_paper.data.preprocess import preprocess_data

import arviz as az
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from aml_working_paper.config import RAW_DATA_DIR, MODELS_DIR, INTERIM_DATA_DIR

In [ ]:
df_aml = pd.read_csv("{}/synthetic_df_aml.csv".format(RAW_DATA_DIR))
### use the draws=1000, tune=1000, chains=6, cores=6, target_accept=0.9, random_seed=42 HMM model
aligned_trace = az.from_netcdf("{}/bayes_hmm_aligned_posterirors.nc".format(MODELS_DIR))

In [6]:
train_df, val_df, test_df, train_sequences, val_sequences, test_sequences, scaler = preprocess_data(df_aml)

In [7]:
aligned_trace.posterior

<xarray.DataTree 'posterior'>
Group: /posterior
    Dimensions:        (chain: 6, draw: 1000, mu_dim_0: 3, mu_dim_1: 3,
                        pi_dim_0: 3, current_dim_0: 3, current_dim_1: 3,
                        sigma_dim_0: 3, sigma_dim_1: 3)
    Coordinates:
      * chain          (chain) int64 48B 0 1 2 3 4 5
      * draw           (draw) int64 8kB 0 1 2 3 4 5 6 ... 994 995 996 997 998 999
      * mu_dim_0       (mu_dim_0) int64 24B 0 1 2
      * mu_dim_1       (mu_dim_1) int64 24B 0 1 2
      * pi_dim_0       (pi_dim_0) int64 24B 0 1 2
      * current_dim_0  (current_dim_0) int64 24B 0 1 2
      * current_dim_1  (current_dim_1) int64 24B 0 1 2
      * sigma_dim_0    (sigma_dim_0) int64 24B 0 1 2
      * sigma_dim_1    (sigma_dim_1) int64 24B 0 1 2
    Data variables:
        mu             (chain, draw, mu_dim_0, mu_dim_1) float64 432kB ...
        pi             (chain, draw, pi_dim_0) float64 144kB ...
        current        (chain, draw, current_dim_0, current_dim_1) float64 432kB ...
        sigma          (chain, draw, sigma_dim_0, sigma_dim_1) float64 432kB ...
    Attributes:
        created_at:                 2026-11-10T17:11:13.690197+00:00
        creation_library:           ArviZ
        creation_library_version:   1.3.1
        creation_library_language:  Python
        inference_library:          pymc
        inference_library_version:  6.3.2
        sample_dims:                ['chain', 'draw']
        sampling_time:              908.4871225357056
        tuning_steps:               1000

In [8]:
pi_draws, current_draws, mu_draws, sigma_draws = extract_posterior_draws(aligned_trace)

### 2. Inspect one account

In [9]:
account_ids = sorted(val_df["account_id"].unique())

account_id = account_ids[0]
sequence = val_sequences[0]
#account_times = (val_df.loc[val_df["account_id"] == account_id, "time"].sort_values().to_numpy())

results = filter_posterior_draws(sequence, pi_draws, current_draws, mu_draws, sigma_draws)

print(f"Example account: {account_id}; periods: {len(sequence)}")

Example account: 12; periods: 100


In [10]:
print(results["filtered_probabilities"].shape)
print(results["uncertainty"].shape)
print(results["surprise"].shape)
print(results["information_gain"].shape)

(100, 3)
(100,)
(100,)
(100,)


In [11]:
account_routing_df = sequence_results_to_dataframe(account_id, results)
#account_routing_df["time"] = account_times

In [12]:
account_routing_df

,account_id,time,p_state_0,p_state_1,p_state_2,uncertainty,surprise,information_gain,expected_state_entropy,parameter_disagreement
0,12,0,5.614262e-03,0.009360,0.985026,0.079811,3.776146,10171.916572,0.077622,0.002188
1,12,1,5.456998e-06,0.841088,0.158907,0.398615,2.039000,6993.852828,0.397784,0.000831
2,12,2,3.551734e-06,0.925215,0.074781,0.242017,1.811520,648.429736,0.241283,0.000735
3,12,3,2.561237e-08,0.997298,0.002702,0.017000,0.973840,1108.263145,0.016975,0.000026
4,12,4,4.096338e-07,0.982553,0.017446,0.080040,2.228148,509.284291,0.079870,0.000169
...,...,...,...,...,...,...,...,...,...,...
95,12,95,9.790596e-06,0.952324,0.047666,0.174501,2.330358,300.357731,0.174185,0.000315
96,12,96,2.340487e-08,0.996379,0.003621,0.021819,0.758047,946.946769,0.021790,0.000030
97,12,97,2.813151e-07,0.975576,0.024423,0.104488,2.039816,436.725576,0.104215,0.000274
98,12,98,2.756450e-08,0.996516,0.003484,0.021115,0.485632,834.192559,0.021088,0.000026


In [14]:
fig = make_subplots(rows=2, cols=1,  shared_xaxes=True,  subplot_titles=("Uncertainty", "Surprise"))
fig.add_trace(go.Scatter(x=account_routing_df["time"], y=account_routing_df["uncertainty"], mode="lines", name="Uncertainty"), row=1, col=1,)
fig.add_trace(go.Scatter(x=account_routing_df["time"], y=account_routing_df["surprise"], mode="lines", name="Surprise"), row=2, col=1)
fig.update_layout(height=600)
fig.show()

### 3. Select Investigations

In [15]:
window_size = 4
candidates = build_candidate_windows(account_routing_df, window_size=window_size)

In [22]:
budget = 3
strategy = "uncertainty"  # "uncertainty", "surprise", "combined", "random"

investigations = select_investigations(candidates, budget=budget, strategy=strategy, random_state=42)
print(f"Selected {len(investigations)} of at most {budget} windows")

Selected 3 of at most 3 windows


In [23]:
investigations

,account_id,start_time,end_time,start_idx,end_idx,uncertainty,surprise,routing_score
0,12.0,21.0,24.0,21.0,24.0,0.628158,3.321549,0.628158
1,12.0,76.0,79.0,76.0,79.0,0.621774,6.048607,0.621774
2,12.0,42.0,45.0,42.0,45.0,0.620147,5.124018,0.620147


### 4. Compare strategies

In [17]:
strategies = ["uncertainty", "surprise", "combined", "random"]
selected_by_strategy = {name: select_investigations(candidates, budget=budget, strategy=name, random_state=42) for name in strategies}
summary = pd.DataFrame([
            {"strategy": name,
            "selected_windows": len(windows),
            "mean_uncertainty": windows["uncertainty"].mean(),
            "mean_surprise": windows["surprise"].mean()}
            for name, windows in selected_by_strategy.items()
            ])


In [18]:
summary

,strategy,selected_windows,mean_uncertainty,mean_surprise
0,uncertainty,3,0.623360,4.831391
1,surprise,3,0.258707,5.840513
2,combined,3,0.583320,5.081603
3,random,3,0.179209,1.551961


- Uncertainty: uncertainty of the averaged filtered state probabilities; not direct evidence of AML risk.
- Surprise: negative log predictive density of the new observation; different scale from entropy.
- Combined: average of the percentile ranks of uncertainty and surprise.
- Random: random generated values. Used as baseline.

### 5. Get all routing DF

In [ ]:
all_routing_df = build_routing_dataframe(val_df, val_sequences, pi_draws, current_draws, mu_draws, sigma_draws)
all_candidates = build_candidate_windows(all_routing_df, window_size=20)
all_investigations = select_investigations(all_candidates, budget=20, strategy="uncertainty", random_state=42)
print(f"Accounts: {all_routing_df['account_id'].nunique()}")
print(f"Candidate windows: {len(all_candidates)}")

Accounts: 21
Candidate windows: 1701


### 6. Save routing df

In [21]:
all_routing_df.to_parquet(INTERIM_DATA_DIR / "routing_df.parquet",index=False)